# geoVI Test — Stochastic SFH Recovery

Geometric Variational Inference (geoVI; Frank et al. 2021) is the
primary inference method for high-dimensional stochastic SFH models.
It constructs a coordinate transformation $g(\boldsymbol{\xi}; \bar{\boldsymbol{\xi}})$
that flattens the posterior metric, making the posterior approximately
Gaussian in the transformed space.

This notebook demonstrates geoVI on a **bursty mock galaxy**
($D \approx 137$: 128 GP latent variables + 9 physical parameters),
then compares with MGVI (the linearized variant) and EVI (the
JIT-compiled fast path that starts with MGVI warmup and refines
with nonlinear geoVI samples).

**Key takeaway:** geoVI recovers the bursty SFH and physical
parameters in $\sim$60 s on CPU, producing 200+ posterior samples
without any MCMC tuning.

In [1]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)

from diffsed import (
    Fitter,
    Fixed,
    Model,
    ParamSpec,
    Uniform,
    load_filter_set,
    load_ssp_data,
)

ssp_data = load_ssp_data(
    "../data/ssp_prsc_miles_chabrier_wNE_logGasU-3.0_logGasZ0.0.h5"
)
filters = load_filter_set(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])

## Model + Mock

Bursty star-forming galaxy: $\sigma_{\rm PS} = 2.0$ (factor $\sim$7
fluctuations in SFR), $\tau_{\rm PS} = 20$ Myr (SN feedback timescale).

In [2]:
spec = ParamSpec(
    sfh_dpl_alpha=Uniform(0.5, 3.0),
    sfh_dpl_beta=Uniform(0.5, 3.0),
    sfh_dpl_tau_gyr=Uniform(0.5, 13.0),
    sfh_dpl_log_peak_sfr=Uniform(-1.0, 2.5),
    sfh_field_psd_sigma=Uniform(0.1, 4.0),
    sfh_field_psd_tau_myr=Uniform(1.0, 300.0),
    met_logzsol=Uniform(-2.0, 0.5),
    dust_tau_bc=Uniform(0.0, 2.0),
    dust_tau_diff=Uniform(0.0, 2.0),
    dust_slope=Fixed(-0.7),
    redshift=Fixed(0.1),
    mean_sfh_type=["dpl", "field"],
    n_grid=128,
)
model = Model(spec, ssp_data, filters=filters)

key = jax.random.PRNGKey(2026)
true_params = spec.sample(key)
true_params.update(
    sfh_dpl_alpha=1.0,
    sfh_dpl_beta=1.5,
    sfh_dpl_tau_gyr=8.0,
    sfh_dpl_log_peak_sfr=jnp.log10(30.0),
    sfh_field_psd_sigma=2.0,
    sfh_field_psd_tau_myr=20.0,
    met_logzsol=-0.3,
    dust_tau_bc=0.5,
    dust_tau_diff=0.3,
)
mock = model.mock(true_params, snr=20.0, key=key)
print(f"D = {spec.n_free}, {len(mock.flux_obs)} data points")

W0318 01:18:09.661329 8209597 cpp_gen_intrinsics.cc:74] Empty bitcode string provided for eigen. Optimizations relying on this IR will be disabled.


D = 9, 5 data points


In [3]:
fitter = Fitter(model, mock.flux_obs, mock.noise, data_type="photometry")

## 1. geoVI (nonlinear)

Standard NIFTy geoVI: each KL iteration draws `n_samples` from the
current Gaussian approximation, refines the expansion point, and
updates the posterior metric $\mathcal{M} = \mathbf{J}^T\mathbf{J} + \mathbf{I}$.
After convergence, `n_posterior_samples` cheap draws give the final posterior.

`sample_mode="nonlinear_resample"` (default) uses the full nonlinear
coordinate transformation $g$, giving more accurate samples than the
linear variant.

In [4]:
key1, key = jax.random.split(key)
t0 = time.perf_counter()
result_geovi = fitter.run(
    "geovi",
    n_iterations=15,
    n_samples=6,
    n_posterior_samples=200,
    verbose=False,
    key=key1,
)
t_geovi = time.perf_counter() - t0
print(f"geoVI: {t_geovi:.1f} s, {result_geovi.diagnostics['n_samples']} samples")

assuming the specified inverse covariance is diagonal


assuming a diagonal covariance matrix;
setting `std_inv` to `cov_inv(ones_like(data))**0.5`


<local>/Projects/diffsed/.venv/lib/python3.12/site-packages/nifty8/re/model.py:164: UserWarning: drawing white parameters;
to silence this warning, overload the `init` method
  warn(msg)


OPTIMIZE_KL: Starting 0001


SL: Iteration 0 ⛰:+1.7944e+01 Δ⛰:inf ➽:1.0000e-04


...

MCG: Iteration 5 ⛰:-6.7791e-04 Δ⛰:2.3054e-04 ➽:1.4436e-03 |∇|:1.0869e+00 ➽:2.3395e+00


MCG: Iteration 6 ⛰:-7.5793e-04 Δ⛰:8.0024e-05 ➽:1.4436e-03 |∇|:7.1888e-01 ➽:2.3395e+00


M: →:1.0 ↺:False #∇²:57 |↘|:7.968929e-02 🞋:1.370000e-03
M: Iteration 9 ⛰:+8.529145e+01 Δ⛰:7.723778e-04 🞋:1.000000e-03


OPTIMIZE_KL: Iteration 0015 ⛰:+8.5291e+01
OPTIMIZE_KL: #(Nonlinear sampling steps) (3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3)
OPTIMIZE_KL: #(KL minimization steps) 9
OPTIMIZE_KL: Likelihood residual(s):
'reduced χ²:     3.6±     8.3, avg:    +0.16±    0.66, #dof:      5'

OPTIMIZE_KL: Prior residual(s):
dust_tau_bc             :: 'reduced χ²:    0.82±    0.53, avg:   -0.039±     0.9, #dof:      1'
dust_tau_diff           :: 'reduced χ²:     2.1±     3.1, avg:    -0.68±     1.3, #dof:      1'
met_logzsol             :: 'reduced χ²:     2.2±     3.2, avg:    +0.62±     1.3, #dof:      1'
psd_xi                  :: 'reduced χ²:     1.1±   0.095, avg:    +0.04±    0.14, #dof:    128'
sfh_dpl_alpha           :: 'reduced χ²:     2.3±     2.7, avg:    +0.32±     1.5, #dof:      1'
sfh_dpl_beta            :: 'reduced χ²:     1.3±     1.5, avg:    -0.26±     1.1, #dof:      1'
sfh_dpl_log_peak_sfr    :: 'reduced χ²:     1.1±     2.2, avg:    +0.58±    0.87, #dof:      1'
sfh_dpl_tau_gyr         :: 'r

geoVI: 43.8 s, 212 samples


## 2. MGVI (linear)

MGVI drops the nonlinear correction and approximates the posterior
directly as $\mathcal{N}(\bar{\boldsymbol{\xi}},\, \mathcal{M}^{-1})$.
Cheaper per iteration, but less accurate for non-Gaussian posteriors.
The `"mgvi"` method is just `"geovi"` with `sample_mode="linear_resample"`.

In [5]:
key2, key = jax.random.split(key)
t0 = time.perf_counter()
result_mgvi = fitter.run(
    "mgvi",
    n_iterations=15,
    n_samples=6,
    n_posterior_samples=200,
    verbose=False,
    key=key2,
)
t_mgvi = time.perf_counter() - t0
print(f"MGVI: {t_mgvi:.1f} s, {result_mgvi.diagnostics['n_samples']} samples")

assuming the specified inverse covariance is diagonal


assuming a diagonal covariance matrix;
setting `std_inv` to `cov_inv(ones_like(data))**0.5`


OPTIMIZE_KL: Starting 0001


SL: Iteration 0 ⛰:+6.7618e+01 Δ⛰:inf ➽:1.0000e-04


SL: Iteration 0 ⛰:-6.1483e+01 Δ⛰:inf ➽:1.0000e-04


...

MCG: Iteration 8 ⛰:-3.9143e+00 Δ⛰:1.0927e+00 ➽:1.2807e-01 |∇|:3.1837e+01 ➽:4.0635e+01


M: →:0.25 ↺:False #∇²:62 |↘|:3.092746e+00 🞋:1.370000e-03
M: Iteration 10 ⛰:+1.025911e+02 Δ⛰:1.390118e+00 🞋:1.000000e-03


M: Iteration Limit Reached!


OPTIMIZE_KL: Iteration 0015 ⛰:+1.0259e+02
OPTIMIZE_KL: Linear sampling status (0, 0, 0, 0, 0, 0)
OPTIMIZE_KL: #(KL minimization steps) 10
OPTIMIZE_KL: Likelihood residual(s):
'reduced χ²:     2.8±     1.7, avg:    -0.26±    0.65, #dof:      5'

OPTIMIZE_KL: Prior residual(s):
dust_tau_bc             :: 'reduced χ²:    0.74±     1.1, avg: +0.00062±    0.86, #dof:      1'
dust_tau_diff           :: 'reduced χ²:    0.27±    0.32, avg:    -0.23±    0.46, #dof:      1'
met_logzsol             :: 'reduced χ²:     3.8±     3.8, avg:     -1.5±     1.2, #dof:      1'
psd_xi                  :: 'reduced χ²:     1.1±   0.094, avg:   -0.064±    0.12, #dof:    128'
sfh_dpl_alpha           :: 'reduced χ²:     1.5±     2.7, avg:    +0.12±     1.2, #dof:      1'
sfh_dpl_beta            :: 'reduced χ²: 1.3e+01±     6.5, avg:     +3.5±    0.91, #dof:      1'
sfh_dpl_log_peak_sfr    :: 'reduced χ²:     9.1±     4.1, avg:     +2.9±    0.69, #dof:      1'
sfh_dpl_tau_gyr         :: 'reduced χ²:     4.1±   

MGVI: 29.4 s, 212 samples


## 3. EVI (JIT-compiled fast path)

EVI is the production workhorse: a fully JIT-compiled loop that
auto-stops when KL converges, with ~500x less Python overhead
than the NIFTy `optimize_kl` path. It starts from MAP automatically.

In [6]:
key3, key = jax.random.split(key)
t0 = time.perf_counter()
result_evi = fitter.run(
    "evi",
    n_iterations=10,
    n_samples=3,
    n_posterior_samples=2000,
    verbose=False,
    key=key3,
)
t_evi = time.perf_counter() - t0
print(f"EVI: {t_evi:.1f} s, {result_evi.diagnostics['n_samples']} samples")

<local>/Projects/diffsed/src/diffsed/fitter.py:915: UserWarning: Seeds disagree: H = 839.1 ± 1515.3 (CV=181%). This may indicate multimodality or poor convergence. Consider increasing n_iterations or inspecting the posterior.
  return self._run_evi_jit(key=key, init_from=init_from, **kwargs)


EVI: 11.9 s, 2000 samples


<local>/Projects/diffsed/src/diffsed/fitter.py:915: UserWarning: Poor fit: chi2/dof=7.7 (expected ~1)
  return self._run_evi_jit(key=key, init_from=init_from, **kwargs)


## SFH Recovery — All Three Methods

In [7]:
sfh_true = model.predict_sfh(true_params)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)

methods = [
    ("geoVI (nonlinear)", result_geovi, "#ff7f0e", t_geovi),
    ("MGVI (linear)", result_mgvi, "#2ca02c", t_mgvi),
    ("EVI (JIT)", result_evi, "#9467bd", t_evi),
]

for ax, (name, result, color, wall) in zip(axes, methods):
    ax.plot(sfh_true["t_gyr"], sfh_true["sfr_full"], "k-", lw=2.5, label="Truth")
    ax.plot(sfh_true["t_gyr"], sfh_true["sfr_mean"],
            "k:", lw=1, alpha=0.4, label="Secular mean")
    model.plot_sfh_posterior(
        result, true_params=true_params, color=color, label=name, ax=ax
    )
    ax.set_xlabel("Lookback time [Gyr]")
    ax.set_title(f"{name} ({wall:.1f} s)")
    ax.set_xlim(0, 13.5)
    ax.legend(fontsize=8)

axes[0].set_ylabel(r"SFR [M$_\odot$ yr$^{-1}$]")
sfr_max = float(np.max(np.array(sfh_true["sfr_full"])))
for ax in axes:
    ax.set_ylim(0, max(3 * sfr_max, 30))

plt.tight_layout()
plt.savefig("figures/test_geovi_sfh.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: figures/test_geovi_sfh.png")

Saved: figures/test_geovi_sfh.png


## Corner Plot — geoVI vs EVI

In [8]:
from diffsed.plotting import safe_corner

param_names = [
    "sfh_dpl_alpha",
    "sfh_dpl_beta",
    "sfh_dpl_tau_gyr",
    "sfh_dpl_log_peak_sfr",
    "sfh_field_psd_sigma",
    "sfh_field_psd_tau_myr",
    "met_logzsol",
    "dust_tau_bc",
    "dust_tau_diff",
]

truths = {k: float(true_params[k]) for k in param_names if k in result_geovi.samples}

fig = safe_corner(result_geovi, params=param_names, truths=truths)
if fig is not None:
    fig.suptitle(f"geoVI Posterior (D ≈ 137, {t_geovi:.1f} s)", y=1.02)
    fig.savefig("figures/test_geovi_corner.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: figures/test_geovi_corner.png")
else:
    print("Corner plot skipped (too few samples?)")

Saved: figures/test_geovi_corner.png


## Photometry Posterior Predictive Check

In [9]:
# Build arrays for posterior-predictive photometry
samples_arr = np.column_stack(
    [np.array(result_geovi.samples[k]) for k in param_names if k in result_geovi.samples]
)
labels = [k for k in param_names if k in result_geovi.samples]

wave_eff = np.array([3551, 4686, 6166, 7480, 8932])  # SDSS ugriz
band_names = ["u", "g", "r", "i", "z"]

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(8, 5), gridspec_kw={"height_ratios": [3, 1]}, sharex=True
)

ax1.errorbar(
    wave_eff, mock.flux_obs, yerr=mock.noise,
    fmt="ko", ms=8, capsize=3, label="Observed", zorder=10,
)

pred_fluxes = []
for i in range(min(len(samples_arr), 50)):
    params_i = dict(true_params)
    for j, name in enumerate(labels):
        params_i[name] = float(samples_arr[i, j])
    if "sfh_field_xi" in result_geovi.samples:
        params_i["sfh_field_xi"] = result_geovi.samples["sfh_field_xi"][i]
    pred = model.predict_photometry(params_i)
    pred_fluxes.append(np.array(pred))
    ax1.plot(wave_eff, pred, "-", color="#ff7f0e", alpha=0.15, lw=0.8)

pred_fluxes = np.array(pred_fluxes)
pred_median = np.median(pred_fluxes, axis=0)

ax1.plot(wave_eff, pred_median, "s-", color="#ff7f0e", ms=6, lw=1.5, label="geoVI median")
ax1.set_ylabel("Flux")
ax1.legend(fontsize=9)
ax1.set_title(f"Photometry Fit — geoVI ({t_geovi:.1f} s)")

residuals = (mock.flux_obs - pred_median) / mock.noise
ax2.bar(wave_eff, residuals, width=300, color="gray", alpha=0.7)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel(r"$(f_{\rm obs} - f_{\rm model}) / \sigma$")
ax2.set_xlabel(r"Wavelength [$\AA$]")
ax2.set_xticks(wave_eff)
ax2.set_xticklabels(band_names)

plt.tight_layout()
plt.savefig("figures/test_geovi_photometry.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: figures/test_geovi_photometry.png")

Saved: figures/test_geovi_photometry.png
